[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-11-lifecycle-hooks.ipynb#scrollTo=jj000001)

---
# Day 11 · Lifecycle Hooks & Custom Graph Adapters
**certified-journeys / hamilton-certified** · Day 11 · Observability

> **Goal for today:** Add timing, logging, and validation to your pipeline **without touching any function code** — using Hamilton's `GraphAdapter` protocol and the built-in `PrintLnHook`.

In [ ]:
%pip install -q sf-hamilton

## The Adapter Pattern

Hamilton separates **what a pipeline computes** (your functions) from **how it's observed** (adapters). This is a key design win — observability code never leaks into business logic.

```
driver.Builder()
    .with_modules(features)        # your functions
    .with_adapters(TimingAdapter()) # cross-cutting observability
    .build()
```

| Hook | When it fires | Use for |
|---|---|---|
| `run_before_graph_execution` | Once, before any node runs | Log pipeline start, validate inputs |
| `run_after_graph_execution` | Once, after all nodes | Log total time, save artifacts |
| `run_before_node_execution` | Before each node | Start per-node timer |
| `run_after_node_execution` | After each node | Log duration, cache output |
| `run_after_node_execution` (error) | If a node raises | Emit alert, increment error counter |

In [ ]:
import sys, types, time, collections
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag
from hamilton.graph_types import HamiltonGraph
from hamilton.lifecycle import GraphExecutionHook, NodeExecutionHook

# ── A simple feature pipeline to instrument ───────────────────────────────────
rng = np.random.default_rng(11)
N = 1000
df = pd.DataFrame({
    'age':    rng.integers(18, 75, N).astype(float),
    'spend':  rng.exponential(80, N),
    'tenure': rng.integers(0, 60, N).astype(float),
})

def age(raw_df: pd.DataFrame) -> pd.Series:
    return raw_df['age']

def spend(raw_df: pd.DataFrame) -> pd.Series:
    return raw_df['spend']

def tenure(raw_df: pd.DataFrame) -> pd.Series:
    return raw_df['tenure']

@tag(feature_type='numerical')
def age_zscore(age: pd.Series) -> pd.Series:
    time.sleep(0.005)  # simulate mild CPU work
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical')
def spend_log(spend: pd.Series) -> pd.Series:
    time.sleep(0.008)
    return np.log1p(spend)

@tag(feature_type='numerical')
def tenure_years(tenure: pd.Series) -> pd.Series:
    time.sleep(0.003)
    return tenure / 12.0

@tag(feature_type='numerical')
def clv_proxy(spend_log: pd.Series, tenure_years: pd.Series) -> pd.Series:
    time.sleep(0.004)
    return spend_log * tenure_years

features_module = types.ModuleType('features')
for fn in [age, spend, tenure, age_zscore, spend_log, tenure_years, clv_proxy]:
    setattr(features_module, fn.__name__, fn)
sys.modules['features'] = features_module

print('Feature pipeline module defined (with simulated latency)')

## Step 1 · Build a Timing Hook

The simplest useful hook: time each node and report the total.

In [ ]:
class TimingHook(NodeExecutionHook, GraphExecutionHook):
    """Log per-node and total pipeline execution time."""

    def __init__(self):
        self._node_start: dict[str, float] = {}
        self.node_times: dict[str, float] = {}
        self._graph_start: float = 0.0
        self.total_time: float = 0.0

    # ── GraphExecutionHook ────────────────────────────────────────────────────
    def run_before_graph_execution(
        self, *, graph: HamiltonGraph, final_vars: list[str],
        inputs: dict, overrides: dict, execution_path: list[str], **kwargs
    ):
        self._graph_start = time.perf_counter()
        print(f'[TimingHook] Pipeline start — requesting: {final_vars}')

    def run_after_graph_execution(
        self, *, graph: HamiltonGraph, results: dict | None,
        success: bool, error: Exception | None, **kwargs
    ):
        self.total_time = time.perf_counter() - self._graph_start
        status = 'OK' if success else f'FAILED: {error}'
        print(f'[TimingHook] Pipeline done in {self.total_time*1000:.1f}ms — {status}')

    # ── NodeExecutionHook ─────────────────────────────────────────────────────
    def run_before_node_execution(
        self, *, node_name: str, node_tags: dict, **kwargs
    ):
        self._node_start[node_name] = time.perf_counter()

    def run_after_node_execution(
        self, *, node_name: str, node_tags: dict,
        result, success: bool, error: Exception | None, **kwargs
    ):
        elapsed = time.perf_counter() - self._node_start.get(node_name, time.perf_counter())
        self.node_times[node_name] = elapsed
        marker = '✓' if success else '✗'
        print(f'  {marker} {node_name:25s} {elapsed*1000:6.1f}ms')


timing_hook = TimingHook()
dr = (
    driver.Builder()
    .with_modules(features_module)
    .with_adapters(timing_hook)
    .build()
)
result = dr.execute(
    ['age_zscore', 'spend_log', 'tenure_years', 'clv_proxy'],
    inputs={'raw_df': df}
)
print(f'\nSlowest node: {max(timing_hook.node_times, key=timing_hook.node_times.get)}')

### What just happened?
- **`TimingHook`** implements two interfaces: `GraphExecutionHook` (fires once per pipeline run) and `NodeExecutionHook` (fires once per node).
- The feature functions themselves are **unchanged** — zero observability code inside business logic.
- `with_adapters(timing_hook)` is all it takes to wire it in.
- Per-node timings let you identify bottlenecks without profiling tools.

## Step 2 · Schema Validation Hook

Run a contract check after every node: detect nulls, wrong types, or out-of-range values as soon as they appear.

In [ ]:
class SchemaValidationHook(NodeExecutionHook):
    """After each node, run registered contracts against the output Series."""

    def __init__(self, contracts: dict):
        """
        contracts: {
            'node_name': [callable(result) -> None (raise on failure)]
        }
        """
        self.contracts = contracts
        self.violations: list[str] = []

    def run_before_node_execution(self, **kwargs):
        pass  # nothing to do before

    def run_after_node_execution(
        self, *, node_name: str, result, success: bool, error: Exception | None, **kwargs
    ):
        if not success or node_name not in self.contracts:
            return
        for check in self.contracts[node_name]:
            try:
                check(result)
            except AssertionError as e:
                msg = f'VIOLATION in {node_name}: {e}'
                self.violations.append(msg)
                print(f'  ⚠ {msg}')


def no_nulls(s):
    assert not s.isna().any(), f'found {s.isna().sum()} NaN values'

def is_positive(s):
    assert (s >= 0).all(), f'min value is {s.min():.3f} (expected >= 0)'

def is_zscore(s, tol=0.1):
    assert abs(s.mean()) < tol, f'mean={s.mean():.4f} (expected ~0)'
    assert abs(s.std() - 1.0) < tol, f'std={s.std():.4f} (expected ~1)'


schema_hook = SchemaValidationHook(contracts={
    'age_zscore':   [no_nulls, is_zscore],
    'spend_log':    [no_nulls, is_positive],
    'tenure_years': [no_nulls, is_positive],
    'clv_proxy':    [no_nulls],
})

dr_validated = (
    driver.Builder()
    .with_modules(features_module)
    .with_adapters(schema_hook)
    .build()
)
result_v = dr_validated.execute(
    ['age_zscore', 'spend_log', 'tenure_years', 'clv_proxy'],
    inputs={'raw_df': df}
)
if schema_hook.violations:
    print(f'\n{len(schema_hook.violations)} violations detected!')
else:
    print('\n✓ All schema contracts passed')

### What just happened?
- **`SchemaValidationHook`** runs per-node checks after execution — no assertions inside feature functions.
- `contracts` is a dict mapping node name → list of validator callables. Easy to extend.
- This pattern complements `@check_output`: use `@check_output` for critical invariants, hooks for monitoring non-fatal issues.

## Step 3 · Caching Hook

Cache slow nodes in-memory. On repeated runs, skip recomputation if inputs haven't changed.

In [ ]:
class InMemoryCacheHook(NodeExecutionHook):
    """Skip execution for slow nodes if output was already computed this session."""

    def __init__(self, cache_nodes: set[str]):
        self.cache_nodes = cache_nodes
        self._cache: dict[str, object] = {}
        self._hits = 0
        self._misses = 0

    def run_before_node_execution(self, *, node_name: str, **kwargs):
        pass  # Hamilton doesn't support short-circuit in before hook
              # (use a custom executor for true skip; this demo logs cache state)

    def run_after_node_execution(
        self, *, node_name: str, result, success: bool, **kwargs
    ):
        if success and node_name in self.cache_nodes:
            if node_name in self._cache:
                self._hits += 1
                print(f'  [cache] HIT  {node_name} — would skip on next run')
            else:
                self._cache[node_name] = result
                self._misses += 1
                print(f'  [cache] MISS {node_name} — stored for next run')

    @property
    def hit_rate(self) -> float:
        total = self._hits + self._misses
        return self._hits / total if total > 0 else 0.0


cache_hook = InMemoryCacheHook(cache_nodes={'spend_log', 'age_zscore'})

dr_cached = (
    driver.Builder()
    .with_modules(features_module)
    .with_adapters(cache_hook)
    .build()
)

print('=== First run ===')
dr_cached.execute(['age_zscore', 'spend_log', 'clv_proxy'], inputs={'raw_df': df})

print('\n=== Second run (same inputs) ===')
dr_cached.execute(['age_zscore', 'spend_log', 'clv_proxy'], inputs={'raw_df': df})

print(f'\nCache hit rate: {cache_hook.hit_rate:.0%}')
print('Note: a real caching adapter would short-circuit execution, not just log.')

### What just happened?
- **`InMemoryCacheHook`** tracks which cached nodes have been computed — in a production system you'd integrate with Redis or a feature store.
- The hook doesn't require any changes to the feature functions.
- Hamilton's official `hamilton.caching` module (in newer versions) provides a full caching executor — this hook shows the underlying concept.

## Step 4 · Composing Multiple Hooks

Pass multiple adapters to `.with_adapters()` — all hooks fire in registration order.

In [ ]:
timing_hook2    = TimingHook()
schema_hook2    = SchemaValidationHook(contracts={'age_zscore': [no_nulls, is_zscore]})

dr_multi = (
    driver.Builder()
    .with_modules(features_module)
    .with_adapters(timing_hook2, schema_hook2)  # both adapters active
    .build()
)

_ = dr_multi.execute(
    ['age_zscore', 'spend_log', 'clv_proxy'],
    inputs={'raw_df': df}
)

print(f'\nTotal pipeline time: {timing_hook2.total_time*1000:.1f}ms')
print(f'Schema violations:   {len(schema_hook2.violations)}')
print('\n✓ Multiple adapters composed successfully')

### What just happened?
- **`.with_adapters(a, b, c)`** composes all hooks — each fires independently in order.
- This is the production pattern: timing adapter + schema adapter + Datadog/OpenTelemetry adapter, all composable without coupling.

In [ ]:
# Challenge: write an AlertHook that:
# 1. Fires run_after_node_execution
# 2. If a pd.Series result has more than 5% NaN values, prints a warning
# 3. Tracks total alerts fired in self.alert_count
#
# Then inject bad data (set 10% of age to NaN) and verify the hook fires

# class AlertHook(NodeExecutionHook):
#     def __init__(self, nan_threshold: float = 0.05):
#         self.nan_threshold = nan_threshold
#         self.alert_count = 0
#     ...

print('Implement AlertHook and test with a DataFrame containing 10% NaN ages!')

---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| `GraphExecutionHook` | Fires once per pipeline run (before/after) |
| `NodeExecutionHook` | Fires once per node execution (before/after) |
| `.with_adapters(...)` | Wire hooks into a Driver — compose multiple |
| Zero coupling | Function code never references the hook — hooks read results passively |
| Production use | Replace `print` with Datadog, OpenTelemetry, or custom alerting |

> **Tip:** Hooks are the right place for cross-cutting concerns: logging, timing, caching, alerting. Keep them out of your feature functions so logic stays testable.

---
## What's next
**Day 12** → Materializers: save and load pipeline outputs to disk, S3, or a feature store — without touching your function code.

Mark Day 11 complete in your [tracker](../index.html).